In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

In [ ]:
!git clone --depth 1 \
  https://github.com/huggingface/lerobot.git \
  /content/lerobot

!git -C /content/lerobot fetch --depth 1 origin ae2b57f
!git -C /content/lerobot checkout ae2b57f

In [ ]:
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy

print("SmolVLA import OK")

import libero
import mujoco

print("LIBERO and MuJoCo imports OK")

In [ ]:
import sys
import importlib
from pathlib import Path

# 1. 找到真正的 LeRobot Python 包
LEROBOT_SRC = "/content/lerobot/src"

assert Path(
    LEROBOT_SRC + "/lerobot/datasets"
).is_dir(), "LeRobot 源码路径不正确"

# 2. 将正确源码路径放在模块搜索列表最前面
if LEROBOT_SRC in sys.path:
    sys.path.remove(LEROBOT_SRC)

sys.path.insert(0, LEROBOT_SRC)

# 3. 清理之前可能缓存的错误 lerobot 模块
# 当前是在修复导入问题，避免和其他模型程序同时运行。
for name in list(sys.modules):
    if name == "lerobot" or name.startswith("lerobot."):
        del sys.modules[name]

# 4. 刷新 Python 的模块查找缓存
importlib.invalidate_caches()

# 5. 重新测试导入
import lerobot
from lerobot.datasets import LeRobotDataset, LeRobotDatasetMetadata
from lerobot.policies.act import ACTPolicy
from lerobot.policies import make_pre_post_processors

print("LeRobot 路径:", lerobot.__file__)
print("LeRobotDataset:", LeRobotDataset)
print("ACTPolicy:", ACTPolicy)
print("make_pre_post_processors:", make_pre_post_processors)
print("\n所有模块导入成功！")

In [ ]:
!MUJOCO_GL=egl lerobot-eval \
  --policy.path=lerobot/smolvla_libero \
  --policy.device=cuda \
  --env.type=libero \
  --env.task=libero_object \
  --env.task_ids="[0]" \
  --env.observation_height=256 \
  --env.observation_width=256 \
  --env.control_mode=relative \
  --env.camera_name_mapping='{"agentview_image":"camera1","robot0_eye_in_hand_image":"camera2"}' \
  --eval.batch_size=1 \
  --eval.n_episodes=1 \
  --eval.use_async_envs=false \
  --output_dir=/content/outputs/smolvla_libero_test

In [ ]:
from pathlib import Path
from IPython.display import Video, display

video_path = Path(
    "/content/outputs/smolvla_libero_test/"
    "videos/libero_object_0/eval_episode_0.mp4"
)

print("视频是否存在:", video_path.is_file())

if video_path.is_file():
    print(
        "文件大小:",
        round(video_path.stat().st_size / 1024**2, 2),
        "MiB"
    )

    display(Video(str(video_path), embed=True))
else:
    print("未找到视频，请检查评估输出目录。")

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import os
import shutil

assert os.path.ismount("/content/gdrive_real"), "Google Drive 未挂载"
assert video_path.is_file(), "仿真视频不存在"

destination = Path(
    "/content/gdrive_real/MyDrive/"
    "lerobot_experiments/smolvla_libero/"
    "eval_episode_0.mp4"
)

destination.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(video_path, destination)

assert destination.is_file()
assert destination.stat().st_size == video_path.stat().st_size

print("已复制视频:", destination)

In [ ]:
from google.colab import drive
from pathlib import Path
import os
import shutil

# 1. 确认仿真视频存在
video_path = Path(
    "/content/outputs/smolvla_libero_test/"
    "videos/libero_object_0/eval_episode_0.mp4"
)

assert video_path.is_file(), "仿真视频不存在"

# 2. 使用当前 Notebook 专用的 Google Drive 挂载点
mount_point = Path("/content/gdrive_eval")

if not os.path.ismount(mount_point):

    # 避免把已存在的普通文件目录误当作挂载目录
    if mount_point.exists() and any(mount_point.iterdir()):
        raise RuntimeError("挂载目录已有普通文件，请换一个目录名")

    drive.mount(str(mount_point))

assert os.path.ismount(mount_point), "Google Drive 挂载失败"

# 3. 指定云端保存位置
destination = (
    mount_point
    / "MyDrive"
    / "lerobot_experiments"
    / "smolvla_libero"
    / "eval_episode_0.mp4"
)

destination.parent.mkdir(parents=True, exist_ok=True)

# 4. 复制视频
shutil.copy2(video_path, destination)

# 5. 验证复制结果
assert destination.is_file()
assert destination.stat().st_size == video_path.stat().st_size

print("视频已复制到 Google Drive：", destination)

In [ ]:
from pathlib import Path
import subprocess
from IPython.display import Video, display

src = Path(
    "/content/outputs/smolvla_libero_test/"
    "videos/libero_object_0/eval_episode_0.mp4"
)

dst = src.with_name("eval_episode_0_normal_speed.mp4")

# 将视频时间戳拉长 4 倍，并按 20 fps 输出
subprocess.run([
    "ffmpeg", "-y",
    "-i", str(src),
    "-vf", "setpts=4.0*PTS",
    "-r", "20",
    "-an",
    str(dst),
], check=True, capture_output=True, text=True)

display(Video(str(dst), embed=True))

In [ ]:
from pathlib import Path
import subprocess
from IPython.display import Video, display

src = Path(
    "/content/outputs/smolvla_libero_5episodes/"
    "videos/libero_object_0/eval_episode_3.mp4"
)

dst = src.with_name("eval_episode_0_normal_speed.mp4")

# 将视频时间戳拉长 4 倍，并按 20 fps 输出
subprocess.run([
    "ffmpeg", "-y",
    "-i", str(src),
    "-vf", "setpts=4.0*PTS",
    "-r", "20",
    "-an",
    str(dst),
], check=True, capture_output=True, text=True)

display(Video(str(dst), embed=True))